In [60]:
import joblib
import polars as pl

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

df = pl.read_csv('../backend/data/nfl_features.csv')

# Look at score distribution
scores = pl.concat([
    df.select(pl.col('home_score').alias('score')),
    df.select(pl.col('away_score').alias('score'))
])

df.head()

season,game_type,week,away_team,away_score,home_team,home_score,result,total,overtime,away_moneyline,home_moneyline,spread_line,away_spread_odds,home_spread_odds,total_line,under_odds,over_odds,home_team_win,home_win_pct,home_ppg,home_opp_ppg,away_win_pct,away_ppg,away_opp_ppg
i64,str,i64,str,i64,str,i64,i64,i64,i64,i64,i64,f64,i64,i64,f64,i64,i64,i64,f64,f64,f64,f64,f64,f64
2019,"""REG""",1,"""GB""",10,"""CHI""",3,-7,13,0,160,-178,3.5,-113,103,47.0,102,-113,0,null,null,null,null,null,null
2019,"""REG""",1,"""KC""",40,"""JAX""",26,-14,66,0,-181,163,-3.5,-101,-110,49.0,102,-113,0,null,null,null,null,null,null
2019,"""REG""",1,"""BAL""",59,"""MIA""",10,-49,69,0,-298,263,-7.0,102,-112,41.0,-105,-105,0,null,null,null,null,null,null
2019,"""REG""",1,"""ATL""",12,"""MIN""",28,16,40,0,159,-176,3.5,-115,104,47.0,101,-112,1,null,null,null,null,null,null
2019,"""REG""",1,"""BUF""",17,"""NYJ""",16,-1,33,0,135,-149,2.5,-101,-110,41.0,-110,-100,0,null,null,null,null,null,null


In [61]:
# Create the score buckets
df = df.with_columns([
    # Home scores
    pl.col('home_score').cut(
        breaks=[10, 17, 24, 31],
        labels=['0-10', '11-17', '18-24', '25-31', '32+']
    ).alias('home_score_bucket'),

    # Away scores
    pl.col('away_score').cut(
        breaks=[10, 17, 24, 31],
        labels=['0-10', '11-17', '18-24', '25-31', '32+']
    ).alias('away_score_bucket')
])

print(df.select(['home_score', 'home_score_bucket', 'away_score', 'away_score_bucket']).head(10))

shape: (10, 4)
┌────────────┬───────────────────┬────────────┬───────────────────┐
│ home_score ┆ home_score_bucket ┆ away_score ┆ away_score_bucket │
│ ---        ┆ ---               ┆ ---        ┆ ---               │
│ i64        ┆ cat               ┆ i64        ┆ cat               │
╞════════════╪═══════════════════╪════════════╪═══════════════════╡
│ 3          ┆ 0-10              ┆ 10         ┆ 0-10              │
│ 26         ┆ 25-31             ┆ 40         ┆ 32+               │
│ 10         ┆ 0-10              ┆ 59         ┆ 32+               │
│ 28         ┆ 25-31             ┆ 12         ┆ 11-17             │
│ 16         ┆ 11-17             ┆ 17         ┆ 11-17             │
│ 32         ┆ 32+               ┆ 27         ┆ 25-31             │
│ 30         ┆ 25-31             ┆ 24         ┆ 18-24             │
│ 21         ┆ 18-24             ┆ 20         ┆ 18-24             │
│ 27         ┆ 25-31             ┆ 27         ┆ 25-31             │
│ 13         ┆ 11-17             

In [62]:
home_scores = df.select([
    pl.col('home_ppg').alias('ppg'), # Avg score
    pl.col('home_opp_ppg').alias('opp_ppg'), # Avg allowed
    pl.col('away_opp_ppg').alias('opp_def'), # Opponent's avg allowed
    pl.col('home_score_bucket').alias('score_bucket')
])

away_scores = df.select([
    pl.col('away_ppg').alias('ppg'),
    pl.col('away_opp_ppg').alias('opp_ppg'),
    pl.col('home_opp_ppg').alias('opp_def'),
    pl.col('away_score_bucket').alias('score_bucket')
])

df_scores = pl.concat([home_scores, away_scores]).drop_nulls()

df_scores.head(10)
print(f"Rows: {len(df_scores)}")

Rows: 3006


In [63]:
X_score = df_scores.select(['ppg', 'opp_ppg', 'opp_def']).to_numpy()
y_score = df_scores.select('score_bucket').to_numpy().ravel()

In [64]:
X_score_train, X_score_test, y_score_train, y_score_test = train_test_split(X_score, y_score, test_size=0.2, random_state=42)

score_model = LogisticRegression(max_iter=1000)
score_model.fit(X_score_train, y_score_train)

y_score_pred = score_model.predict(X_score_test)
print(f"Accuracy: {accuracy_score(y_score_test, y_score_pred)}")
print("Classification Report:")
print(classification_report(y_score_test, y_score_pred))

Accuracy: 0.2807308970099668
Classification Report:
              precision    recall  f1-score   support

        0-10       0.50      0.02      0.04        87
       11-17       0.28      0.07      0.11       121
       18-24       0.27      0.76      0.40       157
       25-31       0.34      0.19      0.24       145
         32+       0.27      0.14      0.19        92

    accuracy                           0.28       602
   macro avg       0.33      0.23      0.19       602
weighted avg       0.32      0.28      0.22       602



In [65]:
# Testing to see distribution of score buckets
print(df_scores.group_by('score_bucket').len().sort('score_bucket'))

shape: (5, 2)
┌──────────────┬─────┐
│ score_bucket ┆ len │
│ ---          ┆ --- │
│ cat          ┆ u32 │
╞══════════════╪═════╡
│ 0-10         ┆ 388 │
│ 11-17        ┆ 595 │
│ 18-24        ┆ 798 │
│ 25-31        ┆ 691 │
│ 32+          ┆ 534 │
└──────────────┴─────┘


In [66]:
# Save model to file


joblib.dump(score_model, '../Backend/model/nfl_score_model.joblib')

['../Backend/model/nfl_score_model.joblib']